In [1]:
import pandas as pd

In [6]:
df = pd.read_csv("../data/cleaned_df.csv")

In [7]:
df

,Unnamed: 0.1,Unnamed: 0,date,text_cleaned,topic_split
0,0,0,"April 1, 2021",War in Afghanistan 2001 2021 2021 Afghanis...,"armed conflicts and attacks,arts and culture,b..."
1,1,1,"April 1, 2022",Russo Ukrainian War 2022 Russian invasion of...,"armed conflicts and attacks,disasters and acci..."
2,2,2,"April 1, 2023",In India a new income tax law comes into ...,"business and economics,disasters and accidents..."
3,3,3,"April 1, 2024",Israel Hamas war World Central Kitchen drone...,"armed conflicts and attacks,disasters and acci..."
4,4,4,"April 1, 2025",Middle Eastern crisis Gaza war Israeli inv...,"armed conflicts and attacks,disasters and acci..."
...,...,...,...,...,...
1689,1689,1689,"September 9, 2021",Afghanistan conflict 2021 evacuation from Af...,"armed conflicts and attacks,arts and culture,b..."
1690,1690,1690,"September 9, 2022",Russo Ukrainian War 2022 Russian invasion of...,"armed conflicts and attacks,disasters and acci..."
1691,1691,1691,"September 9, 2023",Russian invasion of Ukraine Zaporizhzhia Nuc...,"armed conflicts and attacks,arts and culture,l..."
1692,1692,1692,"September 9, 2024",Russian invasion of Ukraine Eastern Ukraine ...,"armed conflicts and attacks,business and econo..."


In [10]:
df['date_cleaned'] = pd.to_datetime(df['date'])

In [11]:
df.head()

,Unnamed: 0.1,Unnamed: 0,date,text_cleaned,topic_split,date_cleaned
0,0,0,"April 1, 2021",War in Afghanistan 2001 2021 2021 Afghanis...,"armed conflicts and attacks,arts and culture,b...",2021-04-01
1,1,1,"April 1, 2022",Russo Ukrainian War 2022 Russian invasion of...,"armed conflicts and attacks,disasters and acci...",2022-04-01
2,2,2,"April 1, 2023",In India a new income tax law comes into ...,"business and economics,disasters and accidents...",2023-04-01
3,3,3,"April 1, 2024",Israel Hamas war World Central Kitchen drone...,"armed conflicts and attacks,disasters and acci...",2024-04-01
4,4,4,"April 1, 2025",Middle Eastern crisis Gaza war Israeli inv...,"armed conflicts and attacks,disasters and acci...",2025-04-01


In [12]:
a = pd.DataFrame(df.groupby(['date_cleaned', 'topic_split'])['text_cleaned'].sum())

a = a.reset_index()
a

,date_cleaned,topic_split,text_cleaned
0,2007-11-03,armed conflicts and attacks,2007 Pakistani state of emergency President ...
1,2010-06-02,"armed conflicts and attacks,arts and culture,d...",The crew of the Libyan M V Rim takes back ...
2,2010-06-03,"other current events,armed conflicts and attac...",British Airways issues an apology for a photo...
3,2010-06-04,"armed conflicts and attacks,business and econo...",Gaza flotilla raid Anti Israel protests tak...
4,2010-06-05,"disasters and accidents,international relation...",A cap is placed on the leaking pipe of the De...
...,...,...,...
1689,2025-10-02,"armed conflicts and attacks,disasters and acci...",Gaza war Israeli invasion of the Gaza Strip ...
1690,2025-10-03,"armed conflicts and attacks,arts and culture,d...",Gaza war Donald Trump s September 2025 Gaza ...
1691,2025-10-04,"armed conflicts and attacks,arts and culture,d...",Gaza war Israeli invasion of the Gaza Strip ...
1692,2025-10-05,"armed conflicts and attacks,disasters and acci...",Gaza war Israeli invasion of the Gaza Strip ...


In [9]:
a = pd.DataFrame(df.groupby(['date_cleaned', 'topic_split']).text_cleaned.count())
a = a.reset_index()
a = a.rename({"text_cleaned": "count"}, axis=1)
a

KeyError: 'date_cleaned'

In [5]:
a.to_csv("news_data_ready_to_plot.csv", index=False)

NameError: name 'a' is not defined

In [8]:
import matplotlib.pyplot as plt

In [ ]:
plt.scatter(df.date_cleaned, df.)

In [2]:
import joblib 
import pandas as pd
a = joblib.load("new_df.pkl")
b = pd.read_csv("cleaned_df.csv")

In [6]:
b.groupby(['date', 'text_cleaned']).agg('topic_split'==str('topic_split'), ','.join)

TypeError: 'bool' object is not callable

In [10]:
c = pd.DataFrame(b.groupby(['date', 'text_cleaned'])['topic_split'].agg(','.join))
c = c.reset_index()
c


,date,text_cleaned,topic_split
0,"April 1, 2021",War in Afghanistan 2001 2021 2021 Afghanis...,"armed conflicts and attacks,arts and culture,b..."
1,"April 1, 2022",Russo Ukrainian War 2022 Russian invasion of...,"armed conflicts and attacks,disasters and acci..."
2,"April 1, 2023",In India a new income tax law comes into ...,"business and economics,disasters and accidents..."
3,"April 1, 2024",Israel Hamas war World Central Kitchen drone...,"armed conflicts and attacks,disasters and acci..."
4,"April 1, 2025",Middle Eastern crisis Gaza war Israeli inv...,"armed conflicts and attacks,disasters and acci..."
...,...,...,...
1689,"September 9, 2021",Afghanistan conflict 2021 evacuation from Af...,"armed conflicts and attacks,arts and culture,b..."
1690,"September 9, 2022",Russo Ukrainian War 2022 Russian invasion of...,"armed conflicts and attacks,disasters and acci..."
1691,"September 9, 2023",Russian invasion of Ukraine Zaporizhzhia Nuc...,"armed conflicts and attacks,arts and culture,l..."
1692,"September 9, 2024",Russian invasion of Ukraine Eastern Ukraine ...,"armed conflicts and attacks,business and econo..."


In [11]:
!pwd

/Users/thomasnguyen/news_project/data


In [12]:
c.to_csv("cleaned_df.csv")

In [14]:
len(b.topic_split.unique())

14

# improving the model

Maybe I can use a pretrained transformer model to classify the news articles

In [17]:
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import scipy.stats as stats 
import numpy as np 
import re
from sklearn.model_selection import train_test_split 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.neighbors import NearestNeighbors
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from xgboost import XGBClassifier
from nltk.corpus import stopwords
import spacy
# ── Data loading & cleaning ───────────────────────────────────────────────────
cleaned_df = pd.read_csv("../data/cleaned_df.csv")
replacements = {
    "culture and entertainment": "other",
    "culture and society": "other",
    ",environment,": ",other,",
    "religion": "other",
    "culture & entertainment": "other",
    "other current events": "other",
    "other and politics": "other",
    ",entertainment": ",other",
}
for old, new in replacements.items():
    cleaned_df['topic_split'] = cleaned_df['topic_split'].str.replace(old, new)

cleaned_df = cleaned_df[~cleaned_df.topic_split.str.contains("other", na=False)]

topic_counts = (
    cleaned_df["topic_split"]
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
)
print(topic_counts)

# ── Features & labels ─────────────────────────────────────────────────────────
X_raw = cleaned_df.copy()
X_raw['text_cleaned'] = (
    X_raw['text_cleaned']
    .str.replace("[^a-zA-Z0-9.]+", " ", regex=True)
    .str.strip()
    .str.lower()
)

y_lists = cleaned_df['topic_split'].apply(lambda x: [i.strip() for i in x.split(',')])
mlb = MultiLabelBinarizer()
y_mlb = mlb.fit_transform(y_lists)
print ("performing MLSMOTE...")
# ── MLSMOTE ───────────────────────────────────────────────────────────────────
def get_minority_samples(X, y):
    label_counts = y.sum(axis=0)
    mean_ir = np.mean(label_counts)
    minority_labels = np.where(label_counts < mean_ir)[0]
    minority_mask = y[:, minority_labels].sum(axis=1) > 0
    return X[minority_mask], y[minority_mask]

def mlsmote(X, y, n_samples=500, k_neighbors=5, random_state=42):
    rng = np.random.default_rng(random_state)
    X_min, y_min = get_minority_samples(X, y)

    if len(X_min) < k_neighbors + 1:
        print(f"Warning: only {len(X_min)} minority samples, reducing k")
        k_neighbors = len(X_min) - 1

    knn = NearestNeighbors(n_neighbors=k_neighbors + 1)
    knn.fit(X_min)

    synthetic_X, synthetic_y = [], []
    for _ in range(n_samples):
        idx = rng.integers(0, len(X_min))
        x_ref, y_ref = X_min[idx], y_min[idx]
        _, neighbor_idxs = knn.kneighbors([x_ref])
        nn_idx = rng.choice(neighbor_idxs[0][1:])
        x_neighbor, y_neighbor = X_min[nn_idx], y_min[nn_idx]
        gap = rng.random()
        x_synthetic = x_ref + gap * (x_neighbor - x_ref)
        y_synthetic = np.logical_or(y_ref, y_neighbor).astype(int)
        synthetic_X.append(x_synthetic)
        synthetic_y.append(y_synthetic)

    return (
        np.vstack([X, np.array(synthetic_X)]),
        np.vstack([y, np.array(synthetic_y)])
    )

# ── FIX 1: Split RAW TEXT first, before any vectorisation ────────────────────
stop_words = list(set(stopwords.words('english')))

splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in splitter.split(X_raw, y_mlb):
    X_train_text = X_raw['text_cleaned'].iloc[train_idx]
    X_test_text  = X_raw['text_cleaned'].iloc[test_idx]
    y_train      = y_mlb[train_idx]
    y_test       = y_mlb[test_idx]

# FIX 2: Fit vectoriser ONLY on training text, then transform both splits
print ("Lemmatizing...")

## Creating a spacy lemmatizer
nlp = spacy.load("en_core_web_sm")
def lemmatize(text):
    doc = nlp(text)
    # Turn it into tokens, ignoring the punctuation
    tokens = [token for token in doc if not token.is_punct]
    # Convert those tokens into lemmas, EXCEPT the pronouns, we'll keep those.
    lemmas = [token.lemma_ if token.pos_ != 'PRON' else token.orth_ for token in tokens]
    return lemmas

print ("Vectorising...")

vectorizer = TfidfVectorizer(
    max_features=5000,   # FIX 3: reduced from 10000 to limit dimensionality
    stop_words=stop_words,
    lowercase=True,
    min_df=10,
    strip_accents='unicode',
    tokenizer=lemmatize
)
X_train = vectorizer.fit_transform(X_train_text).toarray()
X_test  = vectorizer.transform(X_test_text).toarray()   # transform only — no fit

# ── MLSMOTE on training data only ────────────────────────────────────────────
X_train_res, y_train_res = mlsmote(X_train, y_train, n_samples=300, k_neighbors=5)

print(f"Before: {X_train.shape[0]} samples")
print(f"After:  {X_train_res.shape[0]} samples")

before = y_train.sum(axis=0)
after  = y_train_res.sum(axis=0)
print(pd.DataFrame({'before': before, 'after': after}, index=mlb.classes_))

# ── FIX 4: XGBoost with regularisation, and use resampled data ───────────────
print ("Creating the model...")
model = MultiOutputClassifier(
    XGBClassifier(

    )
)

model.fit(X_train_res, y_train_res)  # FIX 9: was using Y_train (typo) and non-resampled data

topic_split
armed conflicts and attacks             1587
law and crime and politics              1389
disasters and accidents                 1374
politics and elections and economics    1198
international relations                 1035
health and environment                   813
business and economics                   671
sports                                   596
science and technology                   462
arts and culture                         344
Name: count, dtype: int64
performing MLSMOTE...
Lemmatizing...
Vectorising...


/opt/miniconda3/envs/news_project/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/opt/miniconda3/envs/news_project/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'ve", 'could', 'far', 'might', 'must', 'need', 'shall', 'win', 'would'] not in stop_words.
  warnings.warn(


Before: 1327 samples
After:  1627 samples
                                      before  after
armed conflicts and attacks             1263   1562
arts and culture                         275    404
business and economics                   537    750
disasters and accidents                 1095   1375
health and environment                   650    864
international relations                  828   1092
law and crime and politics              1111   1404
politics and elections and economics     958   1229
science and technology                   370    507
sports                                   477    664
Creating the model...


,estimator,"XGBClassifier...ree=None, ...)"
,n_jobs,None
,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None


In [18]:
pred = model.predict(X_test)
train_pred = model.predict(X_train_res)


In [19]:
pred

array([[1, 0, 0, ..., 0, 1, 0],
       [1, 0, 1, ..., 1, 0, 0],
       [1, 0, 1, ..., 1, 0, 0],
       ...,
       [1, 0, 1, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 1, 0, 1]])

In [20]:
np.unique(train_pred == y_train_res)

array([ True])

# metrics

In [21]:
from sklearn.metrics import accuracy_score, f1_score, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay


In [22]:
# metrics 
MR = np.all(pred == y_test, axis=1).mean()

In [23]:
MR

0.1783625730994152

In [24]:
loss01 = np.any(y_test != pred, axis=1).mean()

In [25]:
loss01

0.8216374269005848

In [26]:
accuracy_score(pred, y_test)

0.1783625730994152

In [27]:
accuracy_score(train_pred, y_train_res)

1.0

In [28]:
f1_score(y_test, pred, average='micro')

0.8626723223753977

In [29]:
from sklearn.metrics import f1_score
f1_score(y_test, pred, average='macro')

0.7878575248822672

In [30]:
f1_score(y_pred=train_pred, y_true=y_train_res, average='macro')

1.0

In [31]:
f1_score(y_pred=pred, y_true=y_test, average='weighted')

0.8521966211727234

In [281]:
from sklearn.model_selection import cross_val_score

In [282]:
# tuning the model

In [33]:
from sklearn.model_selection import GridSearchCV
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
from sklearn.metrics import hamming_loss

In [ ]:
# bayesian search

from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.multioutput import ClassifierChain
cv_splitter = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Run grid search
opt = BayesSearchCV(
    estimator=ClassifierChain(XGBClassifier()),

    

    search_spaces={
        'estimator__n_estimators': Integer(50, 100),

        'estimator__max_depth': Integer(50, 150),
        "estimator__eta": Real(0.1, 0.3, prior='log-uniform')
    },
    n_iter=32,
    cv=cv_splitter,
    scoring='f1_macro',
    return_train_score=True,  # use macro F1 for imbalanced multilabel
    n_jobs=-1,                # use all CPU cores
    verbose=3
)

opt.fit(X_train_res, y_train_res)




Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV 4/5] END estimator__eta=0.1730799376649326, estimator__lambda=11, estimator__max_depth=57, estimator__n_estimators=52;, score=(train=1.000, test=0.835) total time= 2.7min
[CV 5/5] END estimator__eta=0.1730799376649326, estimator__lambda=11, estimator__max_depth=57, estimator__n_estimators=52;, score=(train=1.000, test=0.831) total time= 2.7min
[CV 1/5] END estimator__eta=0.1730799376649326, estimator__lambda=11, estimator__max_depth=57, estimator__n_estimators=52;, score=(train=1.000, test=0.833) total time= 2.7min
[CV 3/5] END estimator__eta=0.1730799376649326, estimator__lambda=11, estimator__max_depth=57, estimator__n_estimators=52;, score=(train=1.000, test=0.836) total time= 2.7min
[CV 2/5] END estimator__eta=0.1730799376649326, estimator__lambda=11, estimator__max_depth=57, estimator__n_estimators=52;, score=(train=1.000, test=0.838) total time= 2.7min
Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV 4/5

Exception ignored in: <function ResourceTracker.__del__ at 0x115451940>
Traceback (most recent call last):
  File "/opt/miniconda3/envs/news_project/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/miniconda3/envs/news_project/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/miniconda3/envs/news_project/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes


[CV 3/5] END estimator__eta=0.23082384517523266, estimator__lambda=5, estimator__max_depth=50, estimator__n_estimators=67;, score=(train=1.000, test=0.848) total time= 1.9min
[CV 1/5] END estimator__eta=0.23082384517523266, estimator__lambda=5, estimator__max_depth=50, estimator__n_estimators=67;, score=(train=1.000, test=0.845) total time= 2.0min
[CV 5/5] END estimator__eta=0.23082384517523266, estimator__lambda=5, estimator__max_depth=50, estimator__n_estimators=67;, score=(train=1.000, test=0.828) total time= 2.0min
[CV 4/5] END estimator__eta=0.23082384517523266, estimator__lambda=5, estimator__max_depth=50, estimator__n_estimators=67;, score=(train=1.000, test=0.844) total time= 2.0min
[CV 2/5] END estimator__eta=0.23082384517523266, estimator__lambda=5, estimator__max_depth=50, estimator__n_estimators=67;, score=(train=1.000, test=0.844) total time= 2.0min
Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV 1/5] END estimator__eta=0.14592762710850135, estimator__lambda

In [35]:
opt.best_score_


0.8481876782125578

In [251]:
from sklearn.metrics import classification_report
print(classification_report(y_test, opt.predict(X_test), target_names=mlb.classes_))

                                      precision    recall  f1-score   support

         armed conflicts and attacks       0.96      0.99      0.97       324
                    arts and culture       0.77      0.29      0.42        69
              business and economics       0.74      0.65      0.69       134
             disasters and accidents       0.86      0.98      0.91       279
              health and environment       0.94      0.80      0.86       163
             international relations       0.87      0.87      0.87       207
          law and crime and politics       0.84      0.96      0.90       278
politics and elections and economics       0.84      0.85      0.85       240
              science and technology       0.83      0.42      0.56        92
                              sports       0.85      0.82      0.83       119

                           micro avg       0.87      0.85      0.86      1905
                           macro avg       0.85      0.76     

In [292]:
import joblib
joblib.dump(opt, '../models/news_topic_classifier/news_topic_classifier.pkl')
joblib.dump(vectorizer, '../models/news_topic_classifier/tfidf_vectorizer.pkl')
joblib.dump(mlb, '../models/news_topic_classifier/mlb.pkl')

['../models/news_topic_classifier/mlb.pkl']

In [103]:
from sklearn.metrics import hamming_loss, label_ranking_loss

In [291]:
pred = opt.predict(X_test)
print (f"accuracy: {accuracy_score(y_true=y_test, y_pred=pred)}")
print (f"f1 score weighted: {f1_score(y_pred=pred, y_true=y_test, average='weighted')}")
print (f"f1 score micro: {f1_score(y_pred=pred, y_true=y_test, average='micro')}")
print (f"f1 score macro: {f1_score(y_pred=pred, y_true=y_test, average='macro')}")
print (f"hamming loss: {hamming_loss(y_true=y_test, y_pred=pred)}")



accuracy: 0.19005847953216373
f1 score weighted: 0.8489063619056035
f1 score micro: 0.8596352101506741
f1 score macro: 0.7833722316636764
hamming loss: 0.15526315789473685


In [105]:
# metrics 
MR = np.all(pred == y_test, axis=1).mean()
MR

0.21052631578947367

In [176]:
loss01 = np.any(y_test != pred, axis=1).mean()

In [177]:
loss01

0.783625730994152

In [ ]:
import joblib 
joblib.dump(grid_search, '../models/news_topic_classifier/news_topic_classifier.pkl')

['../models/news_topic_classifier.pkl']

In [ ]:
joblib.dump(mlb, "../models/news_topic_classifier/mlb.pkl")

['../models/mlb.pkl']

In [ ]:
joblib.dump(vectorizer, '../models/news_topic_classifier/tfidf_vectorizer.pkl')

['../models/tfidf_vectorizer.pkl']

# trying a neural network

In [14]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
import joblib